In [ ]:
# 05 – Final model: predicting political blocs, merged with the 2027 polls
#
# Why this notebook: notebook 04 showed two big limits when we used the party model for France 2027:
# 1. Alliances: parties merge, split and change names (NUPES in 2022, NFP in 2024, Ensemble…). The
#    database gives them new names, so their history is lost and the model gets confused.
# 2. Parliament vs presidential: our data is about parliament elections, the polls are about the
#    presidential election, where candidates are not parties.
#
# The idea to fix both: instead of predicting each party, we predict each political bloc:
# Far left, Left, Centre, Right, Far right (and Other). A bloc is the sum of the parties of the same political
# family. When the left parties run together (NUPES) or separately (PS, LFI, Greens), the left blocs are the
# same, so their history is kept. When an alliance mixes two blocs (NUPES, NFP: far left + left), we split
# its score between them (step 2). And a presidential candidate can be put in a bloc just like a party, so the
# polls can be merged with our data in the same table.
#
# What this notebook does, in order
# 1. Setup
# 2. Put every party in a bloc (using its political family)
# 3. Build the bloc table: one row per bloc in each election, with its past results
# 4. Train the bloc model (Gradient Boosting, same method as notebook 02/03) and test it on 2015–2023
# 5. Is it better than the party model? Fair comparison on the same test elections
# 6. Check on France 2017 and 2022
# 7. Merge with the polls: same blocs for the 2027 candidates
# 8. Translate the presidential polls into parliament terms (using the 2022 gap)
# 9. Final 2027 estimate, with a range
# 10. How to explain the model, and conclusions
#
# 1. Setup
#
# What this step does: imports the libraries and sets the paths. The polls file goes in the same folder as
# party_elections.csv. The party model from notebook 02 is only used for the comparison in step 5.

import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")

DATA_PATH = Path("/Users/lena/Documents/IRONHACK DATA /Ironhack Data Analysis/WEEK 7/data/party_elections.csv")
POLLS_PATH = DATA_PATH.parent / "sondages-presidentielle-2027.csv"
PARTY_MODEL_PATH = DATA_PATH.parents[1] / "models" / "vote_share_gradient_boosting.joblib"     # notebook 02
BLOC_MODEL_PATH = DATA_PATH.parents[1] / "models" / "vote_share_blocs_gradient_boosting.joblib"  # saved here

TEST_START_YEAR = 2015
INFLATION_CAP = 50
BLOCS = ["Far left", "Left", "Centre", "Right", "Far right"]   # "Other" is kept for training but not shown
GREY, BLUE, ORANGE = "#b4b2a9", "#2a78d6", "#eb6834"

pd.set_option("display.max_columns", 30)

In [ ]:
# 2. Put every party in a bloc
#
# What this step does: gives each party a bloc, using the political family from ParlGov:
# - Far left: Communist/Socialist (radical left: LFI, PCF, LO, NPA…)
# - Left: Social democracy (PS…), Green/Ecologist
# - Centre: Liberal (Renaissance in France, liberal parties elsewhere)
# - Right: Conservative (LR, UDI…), Christian democracy, Agrarian
# - Far right: Right-wing (RN, Reconquête…)
# - Other: Special issue, no family, to be coded (regionalist, single-issue or uncoded parties)
#
# Why families and not names: families don't change when a party changes its name or joins an alliance.
#
# The alliance problem between far left and left: in 2022 ParlGov stored the whole NUPES alliance (LFI,
# PCF, PS, Greens) under one row, "FI". If we put it all in the Far left, the Far left looks huge in 2022 and
# the Left looks almost gone, which is false. There is no official score per party inside the alliance,
# so we split it by the share of the alliance's candidates each bloc had (each party ran under the NUPES
# label in its own constituencies, so it collected roughly its share of the alliance's votes):
# - NUPES 2022 agreement: LFI about 360 constituencies, PCF 50 (far left); Greens about 100, PS 70 (left)
#   → 71% far left, 29% left
# This is an approximation (LFI had more of the strong constituencies), so it is written clearly here.
# The same split rule is used for NFP 2024 in step 9.

FAMILY_TO_BLOC = {
    "Communist/Socialist": "Far left", "Social democracy": "Left", "Green/Ecologist": "Left",
    "Liberal": "Centre",
    "Conservative": "Right", "Christian democracy": "Right", "Agrarian": "Right",
    "Right-wing": "Far right",
}

parties = pd.read_csv(DATA_PATH, parse_dates=["election_date"])
parties["inflation"] = parties["inflation"].clip(upper=INFLATION_CAP)
parties["bloc"] = parties["family_name"].map(FAMILY_TO_BLOC).fillna("Other")

NUPES_2022 = {"Far left": (360 + 50) / 580, "Left": (100 + 70) / 580}           # share of the candidates
ALLIANCE_SPLITS = {("FRA", 2022, "FI"): NUPES_2022}                             # (country, year, row name)


def split_alliances(df, columns):
    """Split each alliance row into one row per bloc, multiplying `columns` by the bloc's share."""
    pieces = [df]
    for (country, year, name), shares in ALLIANCE_SPLITS.items():
        is_alliance = (df["country_name_short"] == country) & (df["year"] == year) & (df["party_name_short"] == name)
        pieces[0] = pieces[0][~is_alliance]
        for bloc, share in shares.items():
            part = df[is_alliance].copy()
            part["bloc"] = bloc
            part[columns] = part[columns] * share
            pieces.append(part)
    return pd.concat(pieces, ignore_index=True)


print(parties["bloc"].value_counts())
france_2022 = split_alliances(parties, ["vote_share"]).query("country_name_short == 'FRA' and year == 2022")
france_2022[["party_name_short", "party_name_english", "family_name", "bloc", "vote_share"]].round(2)


In [ ]:
# How to read the result: 762 party rows are Right, 615 Left, 377 Centre, 297 Far right, 285 Far left and
# 239 Other. In France 2022: the NUPES row ("FI", 25.66%) is now split into 18.14% Far left and 7.52% Left;
# the Ecologists (PE) are Left, Renaissance is Centre, LR and UDI are Right, RN and Reconquête are Far right.
#
# 3. The bloc table
#
# What this step does: adds up the parties of each bloc in each election. One row = one bloc in one
# election, with:
# - vote_share: the bloc's score (the target, what we predict)
# - prev_vote_share, prev2_vote_share: the bloc's score at the previous two elections in that country
#   (taken from the bloc table itself, so alliances and new names don't break the history)
# - prev_change: was the bloc rising or falling?
# - in_government, held_pm: 1 if at least one party of the bloc was in the outgoing government / held the
#   prime minister
# - party_left_right: average left-right position of the bloc's parties (weighted by their votes)
# - n_parties, prev_total_running, economy: the same election information as the party model
#
# For the very first election of each country in our table (1991 or later), the bloc table has no
# previous election, so we use the sum of the parties' previous results stored in notebook 00.


def build_bloc_table(parties):
    """One row per (election, bloc) with the bloc's result, past results and election information."""
    rows = parties.assign(lr_x_votes=parties["party_left_right"] * parties["vote_share"],
                          votes_with_lr=parties["vote_share"].where(parties["party_left_right"].notna()))
    blocs = rows.groupby(["country_name_short", "election_id", "election_date", "year", "bloc"]).agg(
        vote_share=("vote_share", "sum"),
        prev_from_parties=("prev_vote_share", "sum"),
        prev2_from_parties=("prev2_vote_share", "sum"),
        in_government=("in_government", "max"),
        held_pm=("held_pm", "max"),
        lr_x_votes=("lr_x_votes", "sum"),
        votes_with_lr=("votes_with_lr", "sum"),
        n_parties=("n_parties", "first"),
        gdp_growth=("gdp_growth", "first"),
        unemployment=("unemployment", "first"),
        inflation=("inflation", "first"),
    ).reset_index()
    blocs["party_left_right"] = blocs["lr_x_votes"] / blocs["votes_with_lr"].replace(0, np.nan)

    # Full grid: every bloc in every election of the country (score 0 if no party of that bloc ran),
    # so "previous election" really means the previous election, even if the bloc skipped one.
    elections = blocs[["country_name_short", "election_id", "election_date"]].drop_duplicates()
    grid = elections.merge(pd.DataFrame({"bloc": blocs["bloc"].unique()}), how="cross")
    full = grid.merge(blocs[["election_id", "bloc", "vote_share"]], on=["election_id", "bloc"], how="left")
    full["vote_share"] = full["vote_share"].fillna(0)
    full = full.sort_values(["country_name_short", "bloc", "election_date"])
    history = full.groupby(["country_name_short", "bloc"])["vote_share"]
    full["prev_vote_share"] = history.shift(1)
    full["prev2_vote_share"] = history.shift(2)

    blocs = blocs.merge(full[["election_id", "bloc", "prev_vote_share", "prev2_vote_share"]],
                        on=["election_id", "bloc"], how="left")
    blocs["prev_vote_share"] = blocs["prev_vote_share"].fillna(blocs["prev_from_parties"])
    blocs["prev2_vote_share"] = blocs["prev2_vote_share"].fillna(blocs["prev2_from_parties"])
    blocs["prev_change"] = blocs["prev_vote_share"] - blocs["prev2_vote_share"]
    blocs["is_new_bloc"] = (blocs["prev_vote_share"] == 0).astype(int)
    blocs["prev_total_running"] = blocs.groupby("election_id")["prev_vote_share"].transform("sum")
    return blocs.drop(columns=["prev_from_parties", "prev2_from_parties", "lr_x_votes", "votes_with_lr"])


blocs = build_bloc_table(split_alliances(parties, ["vote_share"]))
print(f"{len(blocs)} bloc rows from {blocs['election_id'].nunique()} elections")

france = blocs[(blocs["country_name_short"] == "FRA") & blocs["bloc"].isin(BLOCS)]
france.pivot(index="year", columns="bloc", values="vote_share")[BLOCS].round(1)

In [ ]:
# How to read the table: this is France's history by bloc.
# - Far left: small for a long time (4–13%), then 13.8% in 2017 (Mélenchon's first big score) and 18.1% in
#   2022 (its part of NUPES).
# - Left: around 30% until 2012, then a collapse to 12.2% in 2017 (PS falls from 29% to 7%), 10.2% in 2022.
# - Centre jumps from 2.8% to 28.2% in 2017 (Macron's movement), the Right falls from 48.4% (2007) to 11.3%
#   (2022), and the Far right grows from 4.3% (2007) to 22.9% (2022).
# With parties, the model saw FI jump from 11% to 26% (NUPES) and PS disappear in 2022. With blocs and the
# split, the history stays realistic.
#
# 4. Train the bloc model
#
# What this step does: the same method as the party model, so the results can be compared:
# - same time split: train on 1991–2014, test on 2015–2023
# - same model: Gradient Boosting with absolute-error loss, 400 small trees, learning rate 0.03
# - the depth is chosen by cross-validation (GroupKFold by election) with the same rule as notebook 03:
#   the smallest depth within 1% of the best cross-validation error
# - the bloc name replaces the party family as the category the model knows
#
# The features are known before the election, like before: past results, power, left-right position,
# the election and the economy of the year before.

NUMERIC_FEATURES = ["prev_vote_share", "prev2_vote_share", "prev_change", "is_new_bloc", "in_government",
                    "held_pm", "party_left_right", "n_parties", "prev_total_running",
                    "gdp_growth", "unemployment", "inflation"]
CATEGORICAL_FEATURES = ["bloc"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "vote_share"

train = blocs[blocs["year"] < TEST_START_YEAR]
test = blocs[blocs["year"] >= TEST_START_YEAR]
X_train, y_train, X_test, y_test = train[FEATURES], train[TARGET], test[FEATURES], test[TARGET]
print(f"train: {len(train)} bloc rows | test: {len(test)} bloc rows")

preparation = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
    ("bloc", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])
pipeline = Pipeline([("prep", preparation),
                     ("model", GradientBoostingRegressor(loss="absolute_error", n_estimators=400,
                                                         learning_rate=0.03, random_state=42))])


def simplest_within_1_percent(cv_results):
    """Index of the smallest max_depth whose cross-validation MAE is within 1% of the best one."""
    mae = -np.asarray(cv_results["mean_test_score"])
    depths = np.asarray(cv_results["param_model__max_depth"], dtype=float)
    good_enough = np.flatnonzero(mae <= mae.min() * 1.01)
    return int(good_enough[np.argmin(depths[good_enough])])


search = GridSearchCV(pipeline, {"model__max_depth": [2, 3, 4, 5, 6]}, cv=GroupKFold(n_splits=5),
                      scoring="neg_mean_absolute_error", refit=simplest_within_1_percent, n_jobs=-1)
search.fit(X_train, y_train, groups=train["election_id"])
bloc_model = search.best_estimator_
print("Cross-validation MAE by depth:",
      dict(zip(search.cv_results_["param_model__max_depth"], (-search.cv_results_["mean_test_score"]).round(3))))
print("Chosen depth:", bloc_model.named_steps["model"].max_depth)


def clip_shares(predictions):
    return np.clip(predictions, 0, 100)


def scores(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred), "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R²": r2_score(y_true, y_pred)}


print(f"For comparison, 'same as last time' on the training years: {mean_absolute_error(y_train, train['prev_vote_share']):.2f}")

test = test.assign(bloc_model=clip_shares(bloc_model.predict(X_test)))
pd.DataFrame({"Baseline: same as last time": scores(y_test, test["prev_vote_share"]),
              "Bloc model": scores(y_test, test["bloc_model"])}).T.round(2)

In [ ]:
# How to read the result:
# - Depth: 3 is the best in cross-validation (5.39 points), so it is kept.
# - On the training years (cross-validation), the model is better than "same as last time": 5.39 points
#   against 5.72.
# - But on the 2015–2023 test elections, it is not on the average error: 5.09 points against 4.99 for "same
#   as last time". It only does slightly better on big mistakes (RMSE 6.94 vs 7.03, same R² 0.79).
# - Why: 2015–2023 is a time of big political changes in Europe (new centre parties, far right rising,
#   traditional right and centre-left collapsing). The patterns of 1991–2014 don't help much for the big
#   forces anymore.
# - The errors are bigger than for parties (5 vs 3.6 points for the baseline), because a bloc is bigger
#   than a party, so it moves more points.
#
# 5. Is the bloc model better than the party model?
#
# What this step does: a fair comparison on exactly the same question. We take the party model from
# notebook 02, predict every party in the test elections, and add up its predictions by bloc. Then both
# models are judged on the same thing: the bloc scores of 2015–2023.
# We also save the bloc model, so it can be reused without training again.

party_model = joblib.load(PARTY_MODEL_PATH)
test_parties = parties[parties["year"] >= TEST_START_YEAR].copy()
no_result_two_elections_ago = test_parties["prev2_vote_share"] == 0
test_parties["was_new_last_time"] = (no_result_two_elections_ago & (test_parties["prev_vote_share"] > 0)).astype(int)
test_parties.loc[no_result_two_elections_ago, "prev_change"] = 0
test_parties["n_new_parties"] = test_parties.groupby("election_id")["is_new_party"].transform("sum")
test_parties["prev_total_running"] = test_parties.groupby("election_id")["prev_vote_share"].transform("sum")
test_parties["party_model"] = clip_shares(party_model.predict(test_parties[party_model.feature_names_in_]))
test_parties = split_alliances(test_parties, ["party_model"])        # same NUPES split as the bloc table
party_model_by_bloc = test_parties.groupby(["election_id", "bloc"])["party_model"].sum().rename("party_model_summed")

test = test.merge(party_model_by_bloc, on=["election_id", "bloc"], how="left")
main = test[test["bloc"].isin(BLOCS)]
fair = pd.DataFrame({
    "Baseline: same as last time": scores(main[TARGET], main["prev_vote_share"]),
    "Party model, added up by bloc": scores(main[TARGET], main["party_model_summed"]),
    "Bloc model": scores(main[TARGET], main["bloc_model"]),
}).T
BLOC_MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(bloc_model, BLOC_MODEL_PATH)
fair.round(2)

In [ ]:
# How to read the result: the three methods are judged on the same 5 main blocs in the 2015–2023
# elections, and they are very close: 5.12 (party model added up), 5.20 (same as last time), 5.32 (bloc
# model). The bloc model makes the fewest big mistakes (best RMSE, 7.08), the party model added up has the
# best average error.
# So for the big political forces, in recent years, no method is clearly better than "same as last time".
# The party model's clear gain (3.15 vs 3.59 in notebook 02) comes from small and new parties.
# The bloc model is still the one we need for France, because it follows alliances and can be merged with
# the polls (next steps).
#
# 6. Check on France
#
# What this step does: shows, for the two French test elections (2017 and 2022), the real bloc scores,
# the baseline and the bloc model. 2017 is the year Macron's movement appeared, a very hard case for any
# model; 2022 is the year of NUPES, which broke the party model.

france_test = test[(test["country_name_short"] == "FRA") & test["bloc"].isin(BLOCS)]
france_test[["year", "bloc", TARGET, "prev_vote_share", "bloc_model", "party_model_summed"]].rename(
    columns={TARGET: "real result", "prev_vote_share": "baseline", "bloc_model": "bloc model",
             "party_model_summed": "party model (added up)"}).round(1)

In [ ]:
# How to read the table:
# - 2022, Far left + Left: real 18.1% + 10.2%. The bloc model says 11.6% + 13.6%, the party model added up
#   only 6.6% + 7.0%, because it saw NUPES under the name "FI" and the PS, Greens and PCF as gone. With blocs,
#   the total of the two left blocs is much closer (25.2% vs 28.3% real, against 13.6% for the party model).
#   The bloc model misses the far left's rise inside the left: nothing in history announced it.
# - 2022, Centre: real 25.8%. Bloc model 25.7%, party model 18.2%.
# - 2022, Right and Far right: every method misses. The model expected the Right around 28% and the Far
#   right around 12%, but the Right collapsed (11.3%) and the Far right jumped (22.9%). This kind of
#   political change is not in the data.
# - 2017: everyone misses the Centre (28.2% out of nowhere: Macron's new movement) and the fall of the
#   Left (PS), which was expected at 34% and got 12%.
#
# 7. Merge with the polls
#
# What this step does: puts the 2027 presidential candidates into the same blocs, then merges the polls
# with our bloc table.
# - Far left: LFI (Mélenchon), PCF (Roussel), LO (Arthaud), GRS (Maurel, allied with LFI since 2022)
# - Left: PS, Place Publique (Glucksmann), Les Écologistes (Tondelier)
# - Centre: Renaissance (Attal), Horizons (Philippe)
# - Right: LR (Retailleau, Lisnard…), Debout la France (Dupont-Aignan)
# - Far right: RN (Le Pen), Reconquête (Zemmour)
# - Other: Villepin, Polony
#
# Like in notebook 04: first round only, last 3 months, scores of the same bloc added up in each scenario,
# then averaged so that each poll counts once.

PARTY_TO_BLOC = {
    "LFI": "Far left", "PCF": "Far left", "LO": "Far left", "GRS": "Far left",
    "PS": "Left", "PP": "Left", "LE": "Left",
    "RE": "Centre", "HOR": "Centre",
    "LR": "Right", "DLF": "Right",
    "RN": "Far right", "REC": "Far right",
}

polls = pd.read_csv(POLLS_PATH, parse_dates=["terrain_fin"])
polls["bloc"] = polls["parti"].map(PARTY_TO_BLOC).fillna("Other")
recent_start = polls["terrain_fin"].max() - pd.DateOffset(months=3)
recent = polls[(polls["tour"] == 1) & (polls["terrain_fin"] >= recent_start)]

per_poll = (recent.groupby(["sondage_id", "configuration", "bloc"])["score"].sum().unstack(fill_value=0)
                  .groupby("sondage_id").mean())
polls_2027 = per_poll.mean().rename("polls_2027")
print(f"{len(per_poll)} polls since {recent_start:%d %b %Y}")
polls_2027.round(1)

In [ ]:
# How to read the result: in the presidential polls the Far right (Le Pen + Zemmour) is at 38.0%, the Far
# left (Mélenchon, Roussel, Arthaud, Maurel) at 19.9%, the Centre (Attal + Philippe) at 16.9%, the Left
# (Glucksmann, PS, Tondelier) at 14.1% and the Right (Retailleau or Lisnard + Dupont-Aignan) at 10.8%.
#
# 8. From presidential polls to parliament terms
#
# What this step does: a bloc doesn't get the same score in a presidential and a parliament election. To
# measure the difference, we use 2022, the only year with both elections in our sources:
# - presidential first round, 10 April 2022 (official results): Macron 27.85, Le Pen 23.15, Mélenchon
#   21.95, Zemmour 7.07, Pécresse 4.78, Jadot 4.63, Lassalle 3.13, Roussel 2.28, Dupont-Aignan 2.06,
#   Hidalgo 1.75, Poutou 0.77, Arthaud 0.56
# - parliament first round, June 2022: our bloc table
# The gap = presidential − parliament, for each bloc. Then "polls in parliament terms" = polls − gap.
#
# Careful: it's one election only, so it's a rough translation. But it shows a real pattern: the Far
# right and the Centre do better with a presidential candidate, the Right does better in parliament
# elections, where it has many well-known local candidates.

PRESIDENTIAL_2022 = {
    "Centre": 27.85,                                          # Macron
    "Far right": 23.15 + 7.07,                                # Le Pen, Zemmour
    "Far left": 21.95 + 2.28 + 0.77 + 0.56,                   # Mélenchon, Roussel, Poutou, Arthaud
    "Left": 4.63 + 1.75,                                      # Jadot, Hidalgo
    "Right": 4.78 + 2.06,                                     # Pécresse, Dupont-Aignan
}
parliament_2022 = france[france["year"] == 2022].set_index("bloc")[TARGET]

translation = pd.DataFrame({
    "Presidential 2022": pd.Series(PRESIDENTIAL_2022),
    "Parliament 2022": parliament_2022.reindex(BLOCS),
})
translation["Gap (presidential − parliament)"] = translation["Presidential 2022"] - translation["Parliament 2022"]
translation["Polls 2027 (presidential)"] = polls_2027.reindex(BLOCS)
translation["Polls 2027 in parliament terms"] = (translation["Polls 2027 (presidential)"]
                                                 - translation["Gap (presidential − parliament)"])
translation.loc[BLOCS].round(1)

In [ ]:
# How to read the table: in 2022 the Far left got 7.4 points more in the presidential election than in the
# parliament election (Mélenchon's personal score), the Far right 7.3 more, the Centre 2.1 more. The Left
# got 3.8 points less and the Right 4.4 less: they do better in parliament elections, with many well-known
# local candidates. Once we remove these gaps, the 2027 polls give a parliament-type picture we can compare
# with our model.
# Careful with the two left blocs: their 2022 parliament scores come from our NUPES split by candidates, so
# their gaps are approximate. The total of the two is more reliable than each one.
#
# 9. The final 2027 estimate
#
# What this step does:
# a) builds the 2027 row of each French bloc, like in notebook 04 but by bloc:
#    - last result = the 2024 parliament election (first round, 30 June 2024, Ministry of the Interior),
#      grouped by bloc in the same way as our data (only real party lists, not the small "divers" lists,
#      which ParlGov also leaves out):
#      NFP (Union de la gauche) 28.06, split like NUPES by the share of candidates in the NFP agreement:
#      LFI 229 + PCF 50 (far left) against PS 170 + Écologistes 92 (left) → 52% far left, 48% left
#      Far left = 52% of NFP + FI 0.04 + PCF 0.01
#      Left = 48% of NFP + Écologistes 0.01 + PS 0.09 + PRG 0.04 + other ecologists 0.57
#      Centre = Ensemble 20.04 + Horizons 0.72
#      Right = LR 6.57 + UDI 0.51
#      Far right = RN 29.26 + Ciotti's allies 3.96 + Reconquête 0.75
#    - result before = 2022 (from our bloc table)
#    - in power: the Centre holds the Prime Minister, LR (Right) took part in the 2024–2025 governments
#    - the economy of 2026: approximate values, as in notebook 04 (it barely changes the result)
# b) asks the bloc model for its prediction
# c) gives a range: on the 2015–2023 test elections, we look at how far the model was from reality for big
#    blocs (above 10%). 8 times out of 10 the error was between the two numbers below, so we add them to
#    the prediction to get a range "likely 8 times out of 10".
# d) the final estimate = the average of the model and the polls in parliament terms. The model knows
#    history (power wears parties out, big scores rarely last); the polls know what happened since 2024.
#    Mixing both is what professional forecasters usually do. The 50/50 weight is a simple choice, not
#    something we tested.

NFP_2024 = {"Far left": (229 + 50) / 541, "Left": (170 + 92) / 541}           # share of the candidates
RESULTS_2024 = {
    "Far left": 28.06 * NFP_2024["Far left"] + 0.04 + 0.01,
    "Left": 28.06 * NFP_2024["Left"] + 0.01 + 0.09 + 0.04 + 0.57,
    "Centre": 20.04 + 0.72,
    "Right": 6.57 + 0.51,
    "Far right": 29.26 + 3.96 + 0.75,
}
IN_POWER_2027 = {"Far left": (0, 0), "Left": (0, 0), "Centre": (1, 1), "Right": (1, 0), "Far right": (0, 0)}  # (in_government, held_pm)
ECONOMY_2026 = {"gdp_growth": 0.8, "unemployment": 7.6, "inflation": 1.0}

rows_2027 = france[france["year"] == 2022].set_index("bloc").loc[BLOCS].copy()
rows_2027["prev2_vote_share"] = rows_2027[TARGET]
rows_2027["prev_vote_share"] = pd.Series(RESULTS_2024)
rows_2027["prev_change"] = rows_2027["prev_vote_share"] - rows_2027["prev2_vote_share"]
rows_2027["is_new_bloc"] = 0
rows_2027["in_government"] = [IN_POWER_2027[b][0] for b in rows_2027.index]
rows_2027["held_pm"] = [IN_POWER_2027[b][1] for b in rows_2027.index]
rows_2027["prev_total_running"] = rows_2027["prev_vote_share"].sum()
for column, value in ECONOMY_2026.items():
    rows_2027[column] = value
rows_2027 = rows_2027.reset_index()

big_blocs = test[test["prev_vote_share"] >= 10]
low, high = np.percentile(big_blocs[TARGET] - big_blocs["bloc_model"], [10, 90])
print(f"8 times out of 10, for big blocs, reality was between {low:+.1f} and {high:+.1f} points of the model")

final = pd.DataFrame({
    "2022 result": rows_2027["prev2_vote_share"].to_numpy(),
    "2024 result": rows_2027["prev_vote_share"].to_numpy(),
    "Model 2027": clip_shares(bloc_model.predict(rows_2027[FEATURES])),
}, index=BLOCS)
final["Model range (8 in 10)"] = [f"{max(m + low, 0):.0f}–{m + high:.0f}" for m in final["Model 2027"]]
final["Polls in parliament terms"] = translation.loc[BLOCS, "Polls 2027 in parliament terms"]
final["Final estimate"] = (final["Model 2027"] + final["Polls in parliament terms"]) / 2
final["Final, in presidential terms"] = final["Final estimate"] + translation.loc[BLOCS, "Gap (presidential − parliament)"]
final.round(1)

In [ ]:
# Chart: for each bloc, the 2024 result, the model (with its range), the polls in parliament terms and
# the final estimate.

fig, ax = plt.subplots(figsize=(10, 5))
positions = np.arange(len(BLOCS))
columns = [("2024 result", GREY), ("Model 2027", BLUE), ("Polls in parliament terms", ORANGE),
           ("Final estimate", "#2e2e2e")]
for k, (column, color) in enumerate(columns):
    bars = ax.bar(positions + (k - 1.5) * 0.2, final[column], width=0.19, color=color, label=column)
    ax.bar_label(bars, fmt="%.0f", fontsize=8, padding=2)
ax.errorbar(positions - 0.1, final["Model 2027"], yerr=[np.minimum(-low, final["Model 2027"]), [high] * len(BLOCS)],
            fmt="none", ecolor="#1b4f8a", capsize=4, linewidth=1)
ax.set_xticks(positions, BLOCS)
ax.set(title="France 2027 by bloc: model, polls and final estimate (% of votes, parliament terms)",
       ylabel="% of votes")
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# How to read the table and the chart: grey = 2024, blue = our model (the thin line is its "8 in 10"
# range), orange = the polls translated into parliament terms, black = our final estimate.
# - The model mostly follows 2024 with small moves: Far left 13.5%, Left 15.0%, Centre 19.0%, Right 8.3%,
#   Far right 33.0%. The Centre loses a bit (in power, worn out).
# - Its range is wide (about −9 to +8 points): big political changes can't be predicted from history.
# - The polls in parliament terms are lower for the Centre (14.8%) and higher for the Right (15.2%) and
#   the Left (18.0%).
# - Final estimate (parliament terms): Far right 31.8%, Centre 16.9%, Left 16.5%, Far left 12.9%,
#   Right 11.8%.
# - In presidential terms (adding back the 2022 gap): Far right 39.1%, Far left 20.4%, Centre 19.0%,
#   Left 12.7%, Right 7.3%.
#
# 10. How to explain the model (for the presentation)
#
# In one sentence: the model learns from 30 years of European elections how a political force's score
# usually moves from one election to the next, and applies these patterns to France.
#
# What it looks at, in order of importance:
# - its last result, by far the strongest clue (a bloc at 30% will probably stay around 30%)
# - the result before, to see if it was rising or falling
# - being in power: parties in government usually lose votes (voters blame them)
# - its position (left-right), the number of parties, and how many votes are "up for grabs"
# - the economy of the year before, which matters very little on its own
#
# How it learns (Gradient Boosting): it builds 400 small decision trees, one after another. The first one
# makes a rough guess; each next tree looks at the mistakes left and corrects a little of them. Each tree
# only asks 3 yes/no questions in a row (depth 3, chosen by cross-validation), so no tree can memorise
# individual elections.
#
# How we know it works: it was trained on 1991–2014 and tested on 2015–2023, elections it had never seen,
# It beats "same as last time" on the training years (cross-validation) and makes fewer big mistakes on
# the test years, but its average error on the test years is not smaller (step 4): be honest about it.
#
# What it cannot know: campaigns, candidates, scandals, alliances being made or broken. That's why we
# combine it with the polls for 2027.
#
# Conclusions
# - Predicting blocs fixes the two limits of notebook 04 (with 5 blocs: far left and left are separate).
#   Alliances and new party names no longer break the
#   history, and presidential candidates can be grouped in the same blocs as parties.
# - Accuracy, honestly: on 2015–2023, for the big political forces, no model beats "same as last time" on
#   the average error (5.12 vs 5.20 vs 5.32 points). The bloc model makes the fewest big mistakes and
#   handles alliances much better (France 2022, far left + left: 25.2% predicted vs 28.3% real, against
#   13.6% for the party model).
# - Final models: the party model (notebook 02, 3.15 points) to predict individual parties, and the bloc
#   model (this notebook) for France 2027, because it is the only one that can follow alliances and be
#   merged with the polls.
# - For 2027, our final estimate puts the Far right clearly first (about 32% in a parliament vote, 39% in
#   the presidential first round). Behind, four blocs are close: in a parliament vote Centre 17%, Left
#   16.5%, Far left 13%, Right 12%; in the presidential first round the Far left comes second (20%, mostly
#   Mélenchon), then the Centre (19%), the Left (13%) and the Right (7%).
# - The split between far left and left inside NUPES/NFP is an approximation (share of candidates), so the
#   two left blocs are less certain than the others. Their total (about 29% parliament, 33% presidential)
#   is more reliable.
# - This is an estimate of each bloc's strength, not of who wins. The presidential election is decided in
#   the second round, which our model does not cover.